# 06 — Export artifacts and reproducibility manifest
This notebook collects the outputs generated by the earlier notebooks, records file checksums and creates a ZIP for download. It does not invent missing results; if required outputs are absent, it stops with an explicit error.

In [ ]:
from pathlib import Path
import json, hashlib, zipfile, datetime, os
from google.colab import files
BASE=Path("/content/scifact_study")
required=[
 "dataset_manifest.json",
 "results/clean_rankings.jsonl",
 "results/corrupted_rankings.jsonl",
 "results/recovered_rankings.jsonl",
 "results/retrieval_metrics_summary.csv",
 "results/clean_per_claim_metrics.csv",
 "results/corrupted_per_claim_metrics.csv",
 "results/recovered_per_claim_metrics.csv"
]
missing=[p for p in required if not (BASE/p).exists()]
if missing:
    raise FileNotFoundError("Run earlier notebooks first. Missing: "+", ".join(missing))


In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()
manifest={"created_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
          "project":"SciFact retrieval corruption and reserve-list recovery",
          "status":"executed outputs; inspect notebook outputs and methods before reporting",
          "files":{}}
for p in BASE.rglob("*"):
    if p.is_file() and p.name != "reproducibility_manifest.json":
        manifest["files"][str(p.relative_to(BASE))]={"bytes":p.stat().st_size,"sha256":sha256_file(p)}
(BASE/"reproducibility_manifest.json").write_text(json.dumps(manifest,indent=2))
print("Manifest created with",len(manifest["files"]),"files.")


In [ ]:
zip_path=Path("/content/SciFact_Study_Results.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in BASE.rglob("*"):
        if p.is_file():
            z.write(p,arcname=str(Path("scifact_study")/p.relative_to(BASE)))
print("Created:",zip_path,"bytes:",zip_path.stat().st_size)
files.download(str(zip_path))


## Before submitting
Check that the metrics came from successful notebook runs, include the exact SciFact archive checksum, describe the intervention accurately, and state clearly that no human trust study was conducted.